# `PIIMiddleware`

Middleware used to detect and handle Personally Identifiable Information (PII) in agent conversations.

It can inspect user input, AI output, tool results, and streamed agent events. When PII is found, the middleware can block execution, redact the value, partially mask it, or replace it with a deterministic hash.

- Bases: `AgentMiddleware[AgentState[ResponseT], ContextT, ResponseT]`

## Built-in PII Types

| PII type | Detects |
|---|---|
| `email` | Email addresses |
| `credit_card` | Credit-card numbers that pass Luhn validation |
| `ip` | Valid IPv4 addresses |
| `mac_address` | Colon- or hyphen-separated MAC addresses |
| `url` | `http`/`https` URLs and selected bare URLs |

Custom PII types can also be defined using a regular expression or detector function.

## Constructor

```python
PIIMiddleware(
    pii_type: Literal[
        "email",
        "credit_card",
        "ip",
        "mac_address",
        "url"
    ] | str, # Built-in or custom PII type name
    *,
    strategy: Literal[
        "block",
        "redact",
        "mask",
        "hash"
    ] = "redact", # Action applied when PII is detected
    detector: Callable[
        [str],
        list[PIIMatch]
    ] | str | None = None, # Custom detector or regex
    apply_to_input: bool = True, # Inspect the latest user message
    apply_to_output: bool = False, # Inspect AI output
    apply_to_tool_results: bool = False # Inspect tool-result messages
)
```

## Parameters

* `pii_type` — Name of the sensitive-data type.
  * May be one of the built-in types.
  * May be a custom name when `detector` is provided.

* `strategy` — Determines what happens when matching data is found.
  * Default: `"redact"`
  * Supported values: `"block"`, `"redact"`, `"mask"`, and `"hash"`.

* `detector` — Optional custom detection logic.
  * `None` — Uses the built-in detector registered for `pii_type`.
  * `str` — Treated as a regular-expression pattern.
  * `Callable` — Receives a content string and returns `PIIMatch` objects.

* `apply_to_input` — Whether to inspect the most recent `HumanMessage` before a model call.
  * Default: `True`

* `apply_to_output` — Whether to inspect the most recent `AIMessage` after a model call.
  * Default: `False`
  * Also installs a stream transformer that sanitizes output-side streamed events.

* `apply_to_tool_results` — Whether to inspect `ToolMessage` objects produced after the latest AI tool call.
  * Default: `False`
  * Also installs the output-side stream transformer.

## Attributes

* `pii_type` — Resolved PII type name.
* `strategy` — Resolved handling strategy.
* `detector` — Resolved detector callable.
* `apply_to_input` — Whether user input is inspected.
* `apply_to_output` — Whether AI output is inspected.
* `apply_to_tool_results` — Whether tool results are inspected.
* `transformers` — Contains the internal PII stream transformer when output or tool-result protection is enabled.
* `_resolved_rule` — Internal immutable redaction rule containing the type, strategy, and detector.

## Strategies

### `block`

Raises `PIIDetectionError` as soon as matching data is detected.

```python
PIIMiddleware(
    "credit_card",
    strategy="block"
)
```

### `redact`

Replaces each match with a type-specific placeholder.

```text
saad@example.com
```

becomes:

```text
[REDACTED_EMAIL]
```

### `mask`

Partially hides the value while preserving limited readable context.

Examples:

```text
saad@example.com     -> saad@****.com
4111-1111-1111-1111 -> ****-****-****-1111
192.168.1.25         -> *.*.*.25
AA:BB:CC:DD:EE:FF    -> **:**:**:**:**:FF
https://example.com  -> [MASKED_URL]
```

Custom PII values keep only their last four characters when possible.

### `hash`

Replaces each value with the first eight hexadecimal characters of its SHA-256 digest.

```text
saad@example.com
```

becomes a value in this format:

```text
<email_hash:1a2b3c4d>
```

The same original value produces the same hash placeholder, allowing pseudonymous comparison.

## Properties

### `name`

Returns a middleware name containing its configured PII type.

```python
@property
def name(
    self
) -> str
```

Example:

```text
PIIMiddleware[email]
```

## Methods

1. `_process_content`: Detects and handles PII in one string.
   * Runs the resolved detector.
   * Returns the unchanged content with an empty match list when nothing is found.
   * Applies the configured strategy when matches are found.
   * May raise `PIIDetectionError` when the strategy is `"block"`.
   - **Syntax:**
     ```python
     _process_content(
         self,
         content: str # Content to inspect
     ) -> tuple[
         str, # Sanitized or original content
         list[PIIMatch] # Detected matches
     ]
     ```

2. `before_model`: Checks user input and tool results before model invocation.
   * Inspects the latest `HumanMessage` when `apply_to_input=True`.
   * Finds the latest `AIMessage` and inspects subsequent `ToolMessage` objects when `apply_to_tool_results=True`.
   * Returns a replacement `messages` list only when content was changed.
   * Returns `None` when checks are disabled or no PII is found.
   * Can jump to the agent's `end` node through its hook configuration.
   - **Syntax:**
     ```python
     before_model(
         self,
         state: AgentState[Any], # Current agent state
         runtime: Runtime[ContextT] # LangGraph runtime
     ) -> dict[str, Any] | None
     ```

3. `abefore_model`: Asynchronous version of `before_model`.
   * Uses the same processing logic as the synchronous hook.
   - **Syntax:**
     ```python
     async def abefore_model(
         self,
         state: AgentState[Any], # Current agent state
         runtime: Runtime[ContextT] # LangGraph runtime
     ) -> dict[str, Any] | None
     ```

4. `after_model`: Checks AI output after model invocation.
   * Runs only when `apply_to_output=True`.
   * Inspects the most recent `AIMessage`.
   * Replaces the message content when matching PII is handled.
   * Preserves the message ID, name, and tool calls.
   * Returns `None` when no change is required.
   - **Syntax:**
     ```python
     after_model(
         self,
         state: AgentState[Any], # Current agent state
         runtime: Runtime[ContextT] # LangGraph runtime
     ) -> dict[str, Any] | None
     ```

5. `aafter_model`: Asynchronous version of `after_model`.
   * Uses the same processing logic as the synchronous hook.
   - **Syntax:**
     ```python
     async def aafter_model(
         self,
         state: AgentState[Any], # Current agent state
         runtime: Runtime[ContextT] # LangGraph runtime
     ) -> dict[str, Any] | None
     ```

## `PIIMatch`

Typed dictionary describing one detected sensitive value.

```python
class PIIMatch(TypedDict):
    type: str # PII type name
    value: str # Exact matched value
    start: int # Inclusive start index
    end: int # Exclusive end index
```

### Example

```python
{
    "type": "email",
    "value": "saad@example.com",
    "start": 11,
    "end": 27
}
```

## `PIIDetectionError`

Exception raised when PII is detected while using the `"block"` strategy.

- Bases: `Exception`

## Constructor

```python
PIIDetectionError(
    pii_type: str, # Type of PII that was detected
    matches: Sequence[PIIMatch] # All detected matches
)
```

## Attributes

* `pii_type` — Name of the detected PII type.
* `matches` — List containing all detected matches.

The exception message uses this format:

```text
Detected <count> instance(s) of <pii_type> in text content
```

## Built-in Detector Functions

### `detect_email`

Detects email addresses.

```python
detect_email(
    content: str # Text to scan
) -> list[PIIMatch]
```

### `detect_credit_card`

Detects 16-digit card-number patterns separated by optional spaces or hyphens.

A candidate is returned only when it passes the Luhn checksum.

```python
detect_credit_card(
    content: str # Text to scan
) -> list[PIIMatch]
```

### `detect_ip`

Detects valid IPv4 addresses.

Candidates are validated using Python's `ipaddress` module before being returned.

```python
detect_ip(
    content: str # Text to scan
) -> list[PIIMatch]
```

### `detect_mac_address`

Detects six-pair MAC addresses separated by colons or hyphens.

```python
detect_mac_address(
    content: str # Text to scan
) -> list[PIIMatch]
```

### `detect_url`

Detects:

* URLs beginning with `http://` or `https://`.
* Selected bare URLs that start with `www.` or include a path.

```python
detect_url(
    content: str # Text to scan
) -> list[PIIMatch]
```

## Custom Detection

### Regular-Expression Detector

Pass a regex string for a custom PII type.

```python
middleware = PIIMiddleware(
    "api_key",
    detector=r"sk-[a-zA-Z0-9]{32}",
    strategy="block"
)
```

Every regex match is automatically converted into a `PIIMatch` containing:

```text
type, value, start, end
```

### Callable Detector

A custom function may return matching dictionaries.

```python
from langchain.agents.middleware import PIIMatch, PIIMiddleware

def detect_employee_id(content: str) -> list[PIIMatch]:
    matches: list[PIIMatch] = []

    # Add custom detection logic here.
    return matches

middleware = PIIMiddleware(
    "employee_id",
    detector=detect_employee_id,
    strategy="redact"
)
```

Custom detector results are normalized. A result may use `text` instead of `value`, and may omit `type`; the middleware fills those fields using the configured `pii_type`.

## State-Level Processing

### User Input

When `apply_to_input=True`, the middleware searches backward through the current messages and processes only the latest `HumanMessage`.

```text
Messages before model call
        |
        v
Find latest HumanMessage
        |
        v
Detect configured PII
        |
        v
Apply block/redact/mask/hash
```

### Tool Results

When `apply_to_tool_results=True`, the middleware:

1. Finds the latest `AIMessage`.
2. Processes every `ToolMessage` after it.
3. Replaces only tool messages containing matching PII.

### AI Output

When `apply_to_output=True`, the middleware searches backward for the latest `AIMessage` and processes its content after the model finishes.

## Streaming Protection

When either `apply_to_output` or `apply_to_tool_results` is enabled, the middleware installs an internal `_PIIStreamTransformer`.

The transformer sanitizes PII on streamed wire surfaces, including:

* AI text deltas.
* AI reasoning deltas.
* Streamed and finalized tool-call arguments.
* Tool input events.
* Tool-output deltas.
* Completed tool outputs.
* Tool error messages.
* State snapshots on the `values` stream.
* Legacy full-message streaming events.

The transformer uses a trailing buffer of `128` characters for string deltas. This helps detect PII patterns split across multiple stream chunks before text is released downstream.

For `"block"`, the transformer raises `PIIDetectionError` when a complete match appears in the stream.

State-level `before_model` and `after_model` hooks remain the non-streaming enforcement path.

## Internal Stream Transformer

### `_PIIStreamTransformer`

Internal `StreamTransformer` that mutates streamed protocol events before built-in transformers process them.

- Bases: `StreamTransformer`
- `before_builtins`: `True`
- `required_stream_modes`: `("messages", "tools", "values")`

Important internal methods include:

* `init` — Returns an empty projection state.
* `process` — Routes message, tool, and value events.
* `_process_messages_event` — Sanitizes model message events.
* `_process_tools_event` — Sanitizes tool lifecycle events.
* `_process_values_event` — Sanitizes state snapshots.
* `_redact_value` — Recursively handles string leaves in nested values.
* `_redact_base_message` — Creates a sanitized copy of a message.
* `_mutate_string_field_delta` — Buffers and sanitizes streamed text or reasoning.
* `_mutate_tool_call_chunk_delta` — Sanitizes cumulative tool-call arguments.
* `_finalize_block` — Rechecks finalized content blocks.
* `finalize` — Clears internal stream buffers.
* `fail` — Clears internal buffers after stream failure.

## Exceptions

The constructor raises `ValueError` when:

* `pii_type` is not a built-in type.
* No custom `detector` is supplied for that type.

`PIIDetectionError` is raised whenever matches are found under the `"block"` strategy.

## Examples

### Redact Emails from User Input

```python
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact"
        )
    ]
)
```

### Protect Input and Output

```python
agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
            apply_to_output=True
        )
    ]
)
```

### Use Different Strategies

```python
agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[
        PIIMiddleware(
            "credit_card",
            strategy="mask"
        ),
        PIIMiddleware(
            "url",
            strategy="redact"
        ),
        PIIMiddleware(
            "ip",
            strategy="hash"
        )
    ]
)
```

### Inspect Tool Results

```python
agent = create_agent(
    model="openai:gpt-5.5",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=False,
            apply_to_tool_results=True
        )
    ]
)
```

### Block Custom API Keys

```python
agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block"
        )
    ]
)
```

## Exports

```python
__all__ = [
    "PIIDetectionError",
    "PIIMatch",
    "PIIMiddleware",
    "detect_credit_card",
    "detect_email",
    "detect_ip",
    "detect_mac_address",
    "detect_url",
]
```

## Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/pii.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```